<a href="https://colab.research.google.com/github/Yadav1218/colab/blob/main/group%20project/rakuya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 資料來源 : 樂屋網

# 資料獲取

### 載入套件

In [ ]:
import os
import glob
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

## 爬蟲1(產出rakuyalist資料夾存放rakuyamain.csv)

In [ ]:
table = []
data = {}
existing_ids = set()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

page = 0

for i in range(99):
    i = i + 1
    print(f"Page:{i}")
    time.sleep(1)
    url = "https://www.rakuya.com.tw/rent/rent_search?search=city&city=0&usecode=7%2C8%2C9%2C10&upd=1&page="+str(i)
    try:
        # 發送 HTTP GET 請求
        response = requests.get(url, headers=headers)
        # 確保請求成功
        response.raise_for_status()
        # 取得網頁內容
        content = response.text
        soup = BeautifulSoup(content, "html.parser")
        case = soup.find_all("div", {"class":"obj-info"})
        for list in case:
            case_no = ""
            case_url = ""
            case_price = ""
            case_type = ""
            case_type1 = ""
            case_pattern = ""
            case_size = ""
            case_floor = ""
            case_tfloor = ""
            list1 = list.find("a")
            case_url = list1["href"]
            case_no = case_url.split("=")[-1]
            list2 = list.find("li", {"class":"obj-price"})
            temp00 = list2.find("span")
            case_price = temp00.text.strip().replace("元","").replace(",","")

            list3 = list.find_all("li", {"class":"clearfix"})
            temp01 = list3[0].text.strip()
            case_type = temp01.split("\n")[0].split("/")[0]
            case_type1 = temp01.split("\n")[0].split("/")[-1]
            case_pattern = temp01.split("\n")[-1]

            temp02 = list3[1].text.strip()
            case_size = temp02.split("\n")[0].replace("坪","")
            case_floor = temp02.split("\n")[-1].split("/")[0]
            case_tfloor = temp02.split("\n")[-1].split("/")[-1].replace("樓","")
            # print(case_no, case_url, case_price, case_type, case_type1, case_pattern, case_size, case_floor, case_tfloor)
            data = {
                "物件編號": case_no.strip(),
                "物件網址": case_url.strip(),
                "每月租金": case_price,
                "出租房型": case_type,
                "房屋型態": case_type1,
                "房屋格局": case_pattern,
                "坪數": case_size,
                "樓層": case_floor,
                "總樓層": case_tfloor
            }
            table.append(data)
        #break

    except requests.exceptions.RequestException as e:
        print(f"無法取得網頁內容，錯誤訊息：{e}")

df = pd.DataFrame(table)
# 指定存放 CSV 檔案的資料夾路徑
folder_path = "rakuyalist"
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

filename = "rakuyamain.csv"
output_file = os.path.join(folder_path, filename)
df.to_csv(output_file, encoding="utf-8", sep=',')
print("Finish!")


## 爬蟲2產出rakuyalist000.csv

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
page = 0
table = []
# 指定存放 CSV 檔案的資料夾路徑
folder_path = "rakuyalist"
input_file = os.path.join(folder_path, "rakuyamain.csv")
df = pd.read_csv(input_file)

# print(type(df))
row, col = df.shape
# print(row, col)
for i in range(row):
    case_no = df.values[i][1]
    case_url = df.values[i][2]
    case_price = df.values[i][3]
    case_type = df.values[i][4]
    case_type1 = df.values[i][5]
    case_pattern = df.values[i][6]
    case_size = df.values[i][7]
    case_floor = df.values[i][8]
    case_tfloor = df.values[i][9]
    case_addr = ""
    case_age = ""
    case_cook = 1
    case_pet = 1
    case_guard = 0
    case_money = ""
    case_car = 1
    case_furniture = ""
    url = case_url
    print(i, case_url)
    try:
        # 發送 HTTP GET 請求
        response = requests.get(url, headers=headers)
        # 確保請求成功
        response.raise_for_status()
        # 取得網頁內容
        content = response.text
        soup = BeautifulSoup(content, "html.parser")
        # 地址、屋齡、可否開伙、可否養寵物、管理方式、管理費、傢俱/設備
        list1 = soup.find("h1", {"class":"txt__address"})
        case_addr = list1.text.strip().split("/")[0]
        list2 = soup.find_all("ul", {"class":"group"})
        for list3 in list2:
            list4 = list3.find_all("li")
            for list5 in list4:
                temp = list5.text.strip()
                if "屋齡" in temp:
                    case_age = temp.replace("屋齡","").replace("年","")
                if "開伙" in temp:
                    if "不可" in temp:
                        case_cook = 0
                if "寵物" in temp:
                    if "不可" in temp:
                        case_pet = 0
                if "管理方式" in temp:
                    if "警衛" in temp:
                        case_guard = 1
                if "管理費" in temp:
                    case_money = temp.replace("管理費","")
                if "車位" in temp:
                    if "無車位" in temp:
                        case_car = 0
            list4 = list3.find_all("li", {"class":"is--block"})

            for list5 in list4:
                temp = list5.text.strip().split("\n")
                for i in range(len(temp)):
                    if temp[i] != "設備" and temp[i] != "傢俱":
                        case_furniture += temp[i] + ";"
                        #print(temp[i])
        data = {
                "物件編號": case_no.strip(),
                "物件網址": case_url.strip(),
                "每月租金": case_price,
                "出租房型": case_type,
                "房屋型態": case_type1,
                "房屋格局": case_pattern,
                "坪數": case_size,
                "樓層": case_floor,
                "總樓層": case_tfloor,
                "地址": case_addr,
                "屋齡": case_age,
                "可否開伙": case_cook,
                "否否養寵物": case_pet,
                "管理員": case_guard,
                "管理費": case_money,
                "車位": case_car,
                "傢俱設備": case_furniture
        }
        table.append(data)

        # print(case_addr, case_age, case_cook, case_pet, case_guard, case_money, case_car, case_furniture)
        # break
    except requests.exceptions.RequestException as e:
        print(f"無法取得網頁內容，錯誤訊息：{e}")

    if (i+1) % 100 == 0:
        time.sleep(1)
'''
    if (i+1) % 500 == 0:
        df1 = pd.DataFrame(table)
        s = f"{page:03d}"
        page = page + 1
        fn = "rakuyalist" + s + ".csv"
        df1.to_csv(fn, encoding="utf-8", sep=',')
        table = []
        print(fn)
'''

df1 = pd.DataFrame(table)
s = f"{page:03d}"
page = page + 1
fn = os.path.join(folder_path, "rakuyalist" + s + ".csv")
df1.to_csv(fn, encoding="utf-8", sep=',')
table = []
print(fn)
print("Finish!")



## 爬蟲3(產出rakuyalist-utf8-new.csv)

In [ ]:
# 指定存放 CSV 檔案的資料夾路徑
folder_path = "C:/Users/User/vscode/python/團體專題project/rakuyalist"  # 替換為你的資料夾路徑，例如 "C:/data/csv_files/"

# 找到資料夾中所有 .csv 檔案
input_file = os.path.join(folder_path, "rakuyalist-utf8.csv")

# 用來儲存所有 DataFrame 的列表
table = []
all_set = set()

df = pd.read_csv(input_file)
df['傢俱設備'] = df['傢俱設備'].apply(lambda x: 0 if pd.isna(x) else x)
row, col = df.shape
for i in range(row):
    # case_furniture
    temp_furniture = df.values[i][17]
    if temp_furniture != 0 :
        set_furniture = set(temp_furniture.split(";"))
        if all_set is None:
            all_set = set_furniture
        else:
            all_set = all_set | set_furniture
all_set.remove("")


# print(row, col)
for i in range(row):
    case_no = df.values[i][1]
    case_url = df.values[i][2]
    case_price = df.values[i][3]
    case_type = df.values[i][4]
    case_type1 = df.values[i][5]
    case_pattern = df.values[i][6]
    case_size = df.values[i][7]

    case_floor = df.values[i][8]
    case_tfloor = df.values[i][9]

    temp_addr = df.values[i][10]
    case_addr = temp_addr[0:6]

    case_age = 0
    temp_age = df.values[i][11]
    if "月" in temp_age:
        case_age = 1
    elif "不詳" in temp_age:
        case_age = 99
    else:
        case_age = temp_age

    case_cook = df.values[i][12]
    case_pet = df.values[i][13]
    case_guard = df.values[i][14]

    temp_money = df.values[i][15]
    if type(temp_money) is float:
        case_money = temp_money
    elif "租金內含" in temp_money:
        case_money = 0
    # elif temp_money == "":
    #    case_money = 0
    else:
        case_money = temp_money

    case_car = df.values[i][16]

    data = {
            "物件編號": case_no.strip(),
            "物件網址": case_url.strip(),
            "每月租金": case_price,
            "出租房型": case_type,
            "房屋型態": case_type1,
            "房屋格局": case_pattern,
            "坪數": case_size,
            "樓層": case_floor,
            "總樓層": case_tfloor,
            "地址": case_addr,
            "屋齡": case_age,
            "可否開伙": case_cook,
            "可否養寵物": case_pet,
            "管理員": case_guard,
            "管理費": case_money,
            "車位": case_car
    }
    temp_furniture = df.values[i][17]
    all_list = list(all_set)
    for i in range(len(all_list)):
        data.setdefault(all_list[i], 0)
    if temp_furniture != 0 :
        set_furniture = temp_furniture.strip().split(";")
        # print(set_furniture)
        if set_furniture is not None:
            for i in range(len(set_furniture)):
                if set_furniture[i] != "":
                    data[set_furniture[i]] = 1
    table.append(data)
    # print(table)
    # break

df1 = pd.DataFrame(table)
fn = os.path.join(folder_path, "rakuyalist-utf8-new.csv")
df1.to_csv(fn, encoding="utf-8-sig", sep=',')

read_file = os.path.join(folder_path, "rakuyalist-utf8-new.csv")
output_file = os.path.join(folder_path, "rakuyalist-ansi-new.csv")
with open(read_file, 'r', encoding='utf-8') as infile:
    content = infile.read()
with open(output_file, 'w', encoding='big5', errors='replace') as outfile:
    outfile.write(content)


## 資料前處理

In [ ]:
#載入訓練用資料
import pandas as pd
import numpy as np

#local輸入自己電腦的路徑，以下會統一替代
local="C:/Users/User/vscode/python/團體專題project/"
data = pd.read_csv(local+"rakuyalist-utf8-new.csv")
#去除標題空白格
data.columns = data.columns.str.strip()
#data.head()

## 建立每坪單價(Type=str)，處理成每坪單價(Type=float)，並去除數字中的逗點

In [ ]:
data['每坪單價'] = data['每坪單價'].astype(str).str.replace(',', '', regex=False).astype(float)

## 新增"縣市"及"行政區"欄位

In [ ]:
# 第1~3字設為縣市（例如：台北市、新北市）
data['縣市'] = data['地址'].str[:3]

# 第4~6字設為行政區（例如：信義區、中和區）
data['行政區'] = data['地址'].str[3:]

#print(data[['地址', '縣市', '行政區']].head())

## 新增"郵遞區號"欄位(num)，以便train

In [ ]:
loc_map = {
    # 台北市
    '台北市中正區': 100, '台北市大同區': 103, '台北市中山區': 104, '台北市松山區': 105,
    '台北市大安區': 106, '台北市萬華區': 108, '台北市信義區': 110, '台北市士林區': 111,
    '台北市北投區': 112, '台北市內湖區': 114, '台北市南港區': 115, '台北市文山區': 116,
    # 新北市
    '新北市萬里區': 207, '新北市金山區': 208, '新北市板橋區': 220, '新北市汐止區': 221,
    '新北市深坑區': 222, '新北市石碇區': 223, '新北市瑞芳區': 224, '新北市平溪區': 226,
    '新北市雙溪區': 227, '新北市貢寮區': 228, '新北市新店區': 231, '新北市坪林區': 232,
    '新北市烏來區': 233, '新北市永和區': 234, '新北市中和區': 235, '新北市土城區': 236,
    '新北市三峽區': 237, '新北市樹林區': 238, '新北市鶯歌區': 239, '新北市三重區': 241,
    '新北市新莊區': 242, '新北市泰山區': 243, '新北市林口區': 244, '新北市蘆洲區': 247,
    '新北市五股區': 248, '新北市八里區': 249, '新北市淡水區': 251, '新北市三芝區': 252,
    '新北市石門區': 253
    }
#新增欄位：郵遞區號
data['郵遞區號'] = data['地址'].map(loc_map)

## 產出模型訓練前最後版本(rakuyalist-utf8-new.csv)

In [ ]:
# 儲存為新的 CSV 檔案

data.to_csv(local+"rakuyalist-utf8-new.csv", index=False, encoding='utf-8')

## 切分80/20進行訓練

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("rakuyalist-utf8-new.csv")

print(f"成功讀取原始檔案！總筆數：{df.shape[0]} 筆，欄位數：{df.shape[1]} 個。")

df.columns = df.columns.str.strip()

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("\n--- 資料切分完成 ---")
print(f"訓練集 (Train Set 80%) 維度: {train_df.shape}")
print(f"測試集 (Test Set 20%) 維度: {test_df.shape}")

train_df.to_csv('rakuya_train_80.csv', index=False, encoding='utf-8-sig')
test_df.to_csv('rakuya_test_20.csv', index=False, encoding='utf-8-sig')

print("\n檔案匯出成功！")